In [13]:
import torch
from bermudan import *

In [14]:
set_seed(42)
dev = get_device('cpu')

## Case A: 1D Bermudan put under GBM

In [15]:
diffusion = GBM(r=0.06, sigma=0.2, q=0.0, d=1)
payoff = Put(K=40.0)
option = BermudanOption(diffusion=diffusion, payoff=payoff, r=0.06, T=1.0, N=50)

In [16]:
S0 = torch.tensor([36.0])
M = 100_000

In [17]:
# 1) LSMC
lsmc = LSMC(degree=3)
res = lsmc.price(option, S0, M, device=dev)
print(f'LSMC:  price={res.price:.4f}  std={res.std:.4f}  time={res.elapsed:.2f}s')

LSMC:  price=4.4309  std=0.0099  time=0.49s


In [ ]:
# 2) DOS (small config for quick test)
dos = DOS(hidden_dims=[128, 128], lr=1e-5, n_epochs=50, batch_size=0)
res = dos.price(option, S0, M, device=dev)
print(
    f"DOS:   price={res.price:.4f}  std={res.std:.4f}  time={res.elapsed:.2f}s  params={res.info['total_params']}"
)

In [ ]:
# 3) Policy Gradient (small config for quick test)
pg = PolicyGradient(hidden_dims=[128, 128], lr=1e-5, n_epochs=50, batch_size=50_000, entropy_coeff=0.01)
res = pg.price(option, S0, M, device=dev)
print(f"PG:    price={res.price:.4f}  std={res.std:.4f}  time={res.elapsed:.2f}s  params={res.info['n_params']}")

PG:    price=3.6839  std=0.0084  time=10.43s  params=1217


In [20]:
print('Reference (Longstaff-Schwartz 2001, ATM): ~4.478')

Reference (Longstaff-Schwartz 2001, ATM): ~4.478
